In [1]:
from pathlib import Path

# Memeriksa direktori kerja Jupyter Notebook
BASE = Path.cwd()

print("Direktori kerja saat ini:")
print(BASE)

print("\nIsi direktori:")
for item in sorted(BASE.iterdir()):
    jenis = "Folder" if item.is_dir() else "File"
    print(f"[{jenis}] {item.name}")

Direktori kerja saat ini:
c:\SKRIPSI AKHIR\ALL DATA\GNSS\CDDIS_GNSS Data Daily\HALLOWEEN STORM 2003\Pipeline_Final

Isi direktori:
[File] 01_Pipeline_Final_QC_STEC_Relatif_DOY300.ipynb
[Folder] Output


In [2]:
# Memeriksa folder DOY300 dan file RINEX BAKO

ROOT = BASE.parent

print("Folder proyek:")
print(ROOT)

print("\nFolder di dalam proyek:")
for folder in sorted(ROOT.iterdir()):
    if folder.is_dir():
        print("-", folder.name)

# Mencari file RINEX BAKO DOY300
kandidat_rinex = list(ROOT.glob("**/BAKO/bako3000.03o"))

print("\nKandidat file RINEX:")
for file in kandidat_rinex:
    print(file)

print("\nJumlah kandidat:", len(kandidat_rinex))

Folder proyek:
c:\SKRIPSI AKHIR\ALL DATA\GNSS\CDDIS_GNSS Data Daily\HALLOWEEN STORM 2003

Folder di dalam proyek:
- DOY 300_27 OCT 2003
- DOY 301_28 OCT 2003
- DOY 302_29 OCT 2003
- DOY 303_30 OCT 2003
- DOY 304_31 OCT 2003
- DOY 305_01 NOV 2003
- DOY 306_02 NOV 2003
- DOY 307_03 NOV 2003
- hasil_audit_bako
- Pipeline_Final

Kandidat file RINEX:
c:\SKRIPSI AKHIR\ALL DATA\GNSS\CDDIS_GNSS Data Daily\HALLOWEEN STORM 2003\DOY 300_27 OCT 2003\BAKO\bako3000.03o

Jumlah kandidat: 1


In [3]:
# ==========================================
# KONFIGURASI PIPELINE DOY300 - STASIUN BAKO
# ==========================================

# 1. Menetapkan file RINEX yang akan diproses
FILE_RINEX = kandidat_rinex[0]

# 2. Menetapkan folder utama untuk hasil pengolahan
OUT_BASE = BASE / "output"

# 3. Menetapkan folder khusus tabel dan grafik
OUT_TABEL = OUT_BASE / "tabel"
OUT_GRAFIK = OUT_BASE / "grafik"

# 4. Membuat folder output jika belum tersedia
OUT_TABEL.mkdir(parents=True, exist_ok=True)
OUT_GRAFIK.mkdir(parents=True, exist_ok=True)

# 5. Memeriksa konfigurasi
print("KONFIGURASI PIPELINE")
print("-" * 45)

print("File RINEX:", FILE_RINEX)
print("File tersedia:", FILE_RINEX.exists())
print("Ukuran file:", round(FILE_RINEX.stat().st_size / 1024, 2), "KB")

print("\nFolder output:")
print("Tabel:", OUT_TABEL)
print("Grafik:", OUT_GRAFIK)

KONFIGURASI PIPELINE
---------------------------------------------
File RINEX: c:\SKRIPSI AKHIR\ALL DATA\GNSS\CDDIS_GNSS Data Daily\HALLOWEEN STORM 2003\DOY 300_27 OCT 2003\BAKO\bako3000.03o
File tersedia: True
Ukuran file: 2054.12 KB

Folder output:
Tabel: c:\SKRIPSI AKHIR\ALL DATA\GNSS\CDDIS_GNSS Data Daily\HALLOWEEN STORM 2003\Pipeline_Final\output\tabel
Grafik: c:\SKRIPSI AKHIR\ALL DATA\GNSS\CDDIS_GNSS Data Daily\HALLOWEEN STORM 2003\Pipeline_Final\output\grafik


In [4]:
# ==========================================
# PEMERIKSAAN HEADER RINEX
# ==========================================

# Membuka file RINEX dalam mode baca teks
with FILE_RINEX.open(
    "r",
    encoding="ascii",
    errors="replace"
) as file:

    header_lines = []

    # Membaca file baris demi baris
    for nomor, line in enumerate(file, start=1):

        # Menyimpan baris header
        header_lines.append(line.rstrip("\r\n"))

        # Menampilkan nomor dan isi baris
        print(f"{nomor:03d} | {line.rstrip()}")

        # Berhenti ketika penanda akhir header ditemukan
        if "END OF HEADER" in line:
            break

# Memeriksa apakah akhir header ditemukan
header_valid = any(
    "END OF HEADER" in line
    for line in header_lines
)

print("\nStatus pemeriksaan:")
print("Jumlah baris header:", len(header_lines))
print("END OF HEADER ditemukan:", header_valid)

001 |      2.10           OBSERVATION DATA    G (GPS)             RINEX VERSION / TYPE
002 | teqc  2002Mar14                         20031028 12:06:32UTCPGM / RUN BY / DATE
003 | MSWin2000|IAx86-PII|wcl386 11.0|MSWin95/98/NT/2000|486/DX+  COMMENT
004 | BIT 2 OF LLI FLAGS DATA COLLECTED UNDER A/S CONDITION       COMMENT
005 | bako                                                        MARKER NAME
006 | 23101M002                                                   MARKER NUMBER
007 | eka                 bakosurtanal                            OBSERVER / AGENCY
008 | 2491                TRIMBLE 4000SSE     NP 7.19 / SP 3.04   REC # / TYPE / VERS
009 | 44148               TRM14532.00     NONE                    ANT # / TYPE
010 |  -1836967.8873  6065615.4384  -716260.3127                  APPROX POSITION XYZ
011 |         1.6160        0.0000        0.0000                  ANTENNA: DELTA H/E/N
012 |      1     1                                                WAVELENGTH FACT L1/2
013 |      6

In [5]:
# ==========================================
# PREVIEW BARIS OBSERVASI RINEX
# ==========================================

data_preview = []

with FILE_RINEX.open(
    "r",
    encoding="ascii",
    errors="replace"
) as file:

    # Melewati seluruh header
    for line in file:
        if "END OF HEADER" in line:
            break

    # Membaca 15 baris pertama setelah header
    for nomor, line in enumerate(file, start=1):

        data_preview.append(line.rstrip("\r\n"))

        if nomor == 15:
            break

# Menampilkan hasil pemeriksaan
for nomor, line in enumerate(data_preview, start=1):

    print(f"\nBaris observasi ke-{nomor}")
    print("Panjang karakter:", len(line))
    print("Isi:", repr(line))


Baris observasi ke-1
Panjang karakter: 59
Isi: ' 03 10 27  0  0 30.0000000  0  9G21G 6G 5G25G15G18G30G14G23'

Baris observasi ke-2
Panjang karakter: 63
Isi: '   7272414.07557   5719375.47357  20659955.1484   20659955.7704'

Baris observasi ke-3
Panjang karakter: 15
Isi: '     -3921.0944'

Baris observasi ke-4
Panjang karakter: 63
Isi: '  35449963.44555   2632699.29255  23215464.2584   23215469.9774'

Baris observasi ke-5
Panjang karakter: 15
Isi: '     -3146.2974'

Baris observasi ke-6
Panjang karakter: 63
Isi: '  56683781.12656  17932985.92856  22960277.9694   22960283.2304'

Baris observasi ke-7
Panjang karakter: 15
Isi: '     -3966.5624'

Baris observasi ke-8
Panjang karakter: 63
Isi: '   1091063.23856    872713.18356  22358403.5784   22358406.5554'

Baris observasi ke-9
Panjang karakter: 15
Isi: '     -1224.5944'

Baris observasi ke-10
Panjang karakter: 63
Isi: '  -3502904.17657  -2648798.11957  21209930.3134   21209934.1454'

Baris observasi ke-11
Panjang karakter: 15
Isi: '     

In [6]:
# ==========================================
# 6. PEMERIKSAAN STRUKTUR KOLOM OBSERVASI
# ==========================================

baris_yang_diperiksa = {
    "G21 - baris 1": data_preview[1],
    "G21 - baris 2": data_preview[2],
    "G06 - baris 1": data_preview[3],
    "G06 - baris 2": data_preview[4]
}

for nama, line in baris_yang_diperiksa.items():

    print(f"\n{nama}")
    print("Panjang baris:", len(line))

    for posisi in range(0, len(line), 16):

        field = line[posisi:posisi + 16]

        print(
            f"Kolom {posisi + 1:02d}-{posisi + len(field):02d} | "
            f"Isi: {repr(field)}"
        )


G21 - baris 1
Panjang baris: 63
Kolom 01-16 | Isi: '   7272414.07557'
Kolom 17-32 | Isi: '   5719375.47357'
Kolom 33-48 | Isi: '  20659955.1484 '
Kolom 49-63 | Isi: '  20659955.7704'

G21 - baris 2
Panjang baris: 15
Kolom 01-15 | Isi: '     -3921.0944'

G06 - baris 1
Panjang baris: 63
Kolom 01-16 | Isi: '  35449963.44555'
Kolom 17-32 | Isi: '   2632699.29255'
Kolom 33-48 | Isi: '  23215464.2584 '
Kolom 49-63 | Isi: '  23215469.9774'

G06 - baris 2
Panjang baris: 15
Kolom 01-15 | Isi: '     -3146.2974'


In [7]:
# ==========================================
# 7. VALIDASI OBSERVASI MENGGUNAKAN GEORINEX
# ==========================================

import georinex as gr

# Membaca file RINEX
obs = gr.load(str(FILE_RINEX))

# Memeriksa struktur data
print("Struktur dataset:")
print(obs)

print("\nTipe observasi yang berhasil dibaca:")
print(list(obs.data_vars))

print("\nDimensi dataset:")
print(obs.sizes)

Struktur dataset:
<xarray.Dataset> Size: 4MB
Dimensions:  (time: 2878, sv: 28)
Coordinates:
  * time     (time) datetime64[ns] 23kB 2003-10-27T00:00:30 ... 2003-10-27T23...
  * sv       (sv) <U3 336B 'G01' 'G02' 'G03' 'G04' ... 'G28' 'G29' 'G30' 'G31'
Data variables:
    L1       (time, sv) float64 645kB nan nan nan nan ... nan nan 3.487e+07 nan
    L2       (time, sv) float64 645kB nan nan nan nan ... nan nan 1.001e+07 nan
    C1       (time, sv) float64 645kB nan nan nan nan ... nan nan 5.949e+07 nan
    P2       (time, sv) float64 645kB nan nan nan nan ... nan nan 5.949e+07 nan
    P1       (time, sv) float64 645kB nan nan nan nan nan ... nan nan nan nan
    D1       (time, sv) float64 645kB nan nan nan nan ... nan nan -2.228e+03 nan
Attributes:
    version:          2.1
    interval:         30.0
    rinextype:        obs
    fast_processing:  0
    time_system:      GPS
    filename:         bako3000.03o
    rxmodel:          TRIMBLE 4000SSE NP 7.19 / SP
    position:         [-18

c:\Users\ACER\AppData\Local\Programs\Python\Python313\Lib\site-packages\georinex\obs2.py:58: FutureWarning: In a future version of xarray the default value for join will change from join='outer' to join='exact'. This change will result in the following ValueError: cannot be aligned with join='exact' because index/labels/sizes are not equal along these coordinates (dimensions): 'time' ('time',) The recommendation is to set join explicitly for this case.
  obs = xarray.merge((obs, o))
c:\Users\ACER\AppData\Local\Programs\Python\Python313\Lib\site-packages\georinex\obs2.py:58: FutureWarning: In a future version of xarray the default value for join will change from join='outer' to join='exact'. This change will result in the following ValueError: cannot be aligned with join='exact' because index/labels/sizes are not equal along these coordinates (dimensions): 'sv' ('sv',) The recommendation is to set join explicitly for this case.
  obs = xarray.merge((obs, o))


In [8]:
# Pemeriksaan nilai observasi pada epoch pertama

epoch_pertama = obs.isel(time=0)

print("WAKTU EPOCH PERTAMA")
print(epoch_pertama.time.values)

print("\nJUMLAH SATELIT DENGAN DATA PADA EPOCH PERTAMA")

for tipe in obs.data_vars:
    jumlah = int(epoch_pertama[tipe].notnull().sum().values)
    print(f"{tipe}: {jumlah}")

print("\nNILAI OBSERVASI SETIAP SATELIT")
print(
    epoch_pertama.to_dataframe()
    .dropna(how="all")
    .to_string()
)

WAKTU EPOCH PERTAMA
2003-10-27T00:00:30.000000000

JUMLAH SATELIT DENGAN DATA PADA EPOCH PERTAMA
L1: 9
L2: 9
C1: 9
P2: 9
P1: 0
D1: 9

NILAI OBSERVASI SETIAP SATELIT
                   time            L1            L2            C1            P2  P1        D1
sv                                                                                           
G01 2003-10-27 00:00:30           NaN           NaN           NaN           NaN NaN       NaN
G02 2003-10-27 00:00:30           NaN           NaN           NaN           NaN NaN       NaN
G03 2003-10-27 00:00:30           NaN           NaN           NaN           NaN NaN       NaN
G04 2003-10-27 00:00:30           NaN           NaN           NaN           NaN NaN       NaN
G05 2003-10-27 00:00:30  5.668378e+07  1.793299e+07  2.296028e+07  2.296028e+07 NaN -3966.562
G06 2003-10-27 00:00:30  3.544996e+07  2.632699e+06  2.321546e+07  2.321547e+07 NaN -3146.297
G07 2003-10-27 00:00:30           NaN           NaN           NaN           NaN NaN

In [9]:
# Pemeriksaan struktur fisik record epoch pertama

baris_mentah = []

with FILE_RINEX.open("r", encoding="ascii", errors="replace") as f:
    # Lewati header
    for line in f:
        if "END OF HEADER" in line:
            break

    # Ambil record epoch pertama dan baris-baris setelahnya
    for i in range(19):
        line = f.readline()

        if not line:
            break

        # Hilangkan newline saja, jangan hilangkan spasi
        baris_mentah.append(line.rstrip("\r\n"))

print("JUMLAH BARIS FISIK:", len(baris_mentah))

for nomor, line in enumerate(baris_mentah, start=1):
    print(f"\nBaris fisik ke-{nomor}")
    print("Panjang karakter:", len(line))
    print("Isi mentah:", repr(line))

    if nomor > 1:
        print("Pembagian field 16 karakter:")

        for awal in range(0, len(line), 16):
            akhir = awal + 16
            print(
                f"Kolom {awal+1:02d}-{akhir:02d}:",
                repr(line[awal:akhir])
            )

JUMLAH BARIS FISIK: 19

Baris fisik ke-1
Panjang karakter: 59
Isi mentah: ' 03 10 27  0  0 30.0000000  0  9G21G 6G 5G25G15G18G30G14G23'

Baris fisik ke-2
Panjang karakter: 63
Isi mentah: '   7272414.07557   5719375.47357  20659955.1484   20659955.7704'
Pembagian field 16 karakter:
Kolom 01-16: '   7272414.07557'
Kolom 17-32: '   5719375.47357'
Kolom 33-48: '  20659955.1484 '
Kolom 49-64: '  20659955.7704'

Baris fisik ke-3
Panjang karakter: 15
Isi mentah: '     -3921.0944'
Pembagian field 16 karakter:
Kolom 01-16: '     -3921.0944'

Baris fisik ke-4
Panjang karakter: 63
Isi mentah: '  35449963.44555   2632699.29255  23215464.2584   23215469.9774'
Pembagian field 16 karakter:
Kolom 01-16: '  35449963.44555'
Kolom 17-32: '   2632699.29255'
Kolom 33-48: '  23215464.2584 '
Kolom 49-64: '  23215469.9774'

Baris fisik ke-5
Panjang karakter: 15
Isi mentah: '     -3146.2974'
Pembagian field 16 karakter:
Kolom 01-16: '     -3146.2974'

Baris fisik ke-6
Panjang karakter: 63
Isi mentah: '  566837

In [10]:
# ============================================================
# IDENTIFIKASI EPOCH DENGAN JUMLAH SATELIT TERBANYAK
# ============================================================

jumlah_sat_epoch = obs["L1"].notnull().sum(dim="sv")

indeks_maksimum = int(jumlah_sat_epoch.argmax().values)

waktu_maksimum = obs.time.values[indeks_maksimum]
jumlah_maksimum = int(jumlah_sat_epoch.values[indeks_maksimum])

satelit_maksimum = (
    obs["L1"]
    .isel(time=indeks_maksimum)
    .dropna("sv")["sv"]
    .values
)

print("EPOCH DENGAN JUMLAH SATELIT TERBANYAK")
print("Waktu             :", waktu_maksimum)
print("Jumlah satelit    :", jumlah_maksimum)
print("Daftar satelit    :", list(satelit_maksimum))

print("\nDISTRIBUSI JUMLAH SATELIT PER EPOCH")
print(jumlah_sat_epoch.to_series().value_counts().sort_index())

EPOCH DENGAN JUMLAH SATELIT TERBANYAK
Waktu             : 2003-10-27T00:00:30.000000000
Jumlah satelit    : 9
Daftar satelit    : [np.str_('G05'), np.str_('G06'), np.str_('G14'), np.str_('G15'), np.str_('G18'), np.str_('G21'), np.str_('G23'), np.str_('G25'), np.str_('G30')]

DISTRIBUSI JUMLAH SATELIT PER EPOCH
L1
7     182
8    1301
9    1395
Name: count, dtype: int64


c:\Users\ACER\AppData\Local\Programs\Python\Python313\Lib\site-packages\xarray\core\dataarray.py:6563: FutureWarning: Behaviour of argmin/argmax with neither dim nor axis argument will change to return a dict of indices of each dimension. To get a single, flat index, please use np.argmin(da.data) or np.argmax(da.data) instead of da.argmin() or da.argmax().
  result = self.variable.argmax(dim, axis, keep_attrs, skipna)


In [11]:
# ============================================================
# VALIDASI RECORD MENTAH VS GEORINEX
# EPOCH PERTAMA - SELURUH SATELIT
# ============================================================

# Baca ulang file mentah tanpa mengubah isi baris
raw_lines = FILE_RINEX.read_text(
    encoding="ascii",
    errors="replace"
).splitlines()

# Cari akhir header
idx_header = next(
    i for i, line in enumerate(raw_lines)
    if "END OF HEADER" in line
)

# Ambil epoch pertama
epoch_line = raw_lines[idx_header + 1]

# Ekstraksi daftar satelit dari epoch record
sat_text = epoch_line[32:68]

satelit_list = [
    sat_text[j:j+3].strip().replace(" ", "0")
    for j in range(0, len(sat_text), 3)
    if sat_text[j:j+3].strip()
]

hasil_validasi = []

for posisi, sat in enumerate(satelit_list):

    # Dua baris fisik observasi per satelit
    idx_obs = idx_header + 2 + posisi * 2

    baris_1 = raw_lines[idx_obs].ljust(80)
    baris_2 = raw_lines[idx_obs + 1].ljust(80)

    # Empat field pertama pada baris pertama
    field = [
        baris_1[0:16],
        baris_1[16:32],
        baris_1[32:48],
        baris_1[48:64]
    ]

    # D1 diduga berada pada field pertama baris lanjutan
    field_d1 = baris_2[0:16]

    nilai_mentah = {}

    for nama, isi in zip(
        ["L1", "L2", "C1", "P2"],
        field
    ):
        nilai = isi[:14].strip()
        nilai_mentah[nama] = (
            float(nilai) if nilai else np.nan
        )

    nilai_d1 = field_d1[:14].strip()

    nilai_mentah["D1"] = (
        float(nilai_d1) if nilai_d1 else np.nan
    )

    # Ambil nilai GeoRinex pada epoch pertama
    nilai_georinex = {
        nama: float(obs[nama].isel(time=0).sel(sv=sat).values)
        for nama in ["L1", "L2", "C1", "P2", "D1"]
    }

    for nama in nilai_mentah:

        a = nilai_mentah[nama]
        b = nilai_georinex[nama]

        cocok = (
            np.isclose(a, b, rtol=1e-10, atol=1e-8)
            if np.isfinite(a) and np.isfinite(b)
            else (np.isnan(a) and np.isnan(b))
        )

        hasil_validasi.append({
            "Satelit": sat,
            "Observasi": nama,
            "Nilai_Mentah": a,
            "Nilai_GeoRinex": b,
            "Sesuai": cocok
        })

df_validasi = pd.DataFrame(hasil_validasi)

display(df_validasi)

print("\nRINGKASAN VALIDASI")
print("Jumlah perbandingan:", len(df_validasi))
print("Jumlah sesuai:", df_validasi["Sesuai"].sum())
print("Jumlah tidak sesuai:", (~df_validasi["Sesuai"]).sum())

NameError: name 'np' is not defined

In [ ]:
# ============================================================
# AUDIT EPOCH RECORD SELURUH FILE RINEX
# ============================================================

import re
from collections import Counter

epoch_records = []

for nomor_baris, line in enumerate(raw_lines[idx_header + 1:], 
                                   start=idx_header + 2):

    if len(line) < 32:
        continue

    try:
        yy = int(line[1:3])
        month = int(line[4:6])
        day = int(line[7:9])
        hour = int(line[10:12])
        minute = int(line[13:15])
        second = float(line[15:26])

        epoch_flag = int(line[28:29])
        jumlah_sat = int(line[29:32])

    except (ValueError, IndexError):
        continue

    # Validasi dasar format epoch DOY300
    if not (
        yy == 3
        and month == 10
        and day == 27
        and 0 <= hour <= 23
        and 0 <= minute <= 59
        and 0 <= second < 61
        and 0 <= epoch_flag <= 6
        and 0 <= jumlah_sat <= 12
    ):
        continue

    waktu = (
        pd.Timestamp(
            year=2000 + yy,
            month=month,
            day=day,
            hour=hour,
            minute=minute
        )
        + pd.Timedelta(seconds=second)
    )

    sat_text = line[32:68]

    satelit = [
        sat_text[j:j+3].strip().replace(" ", "0")
        for j in range(0, len(sat_text), 3)
        if sat_text[j:j+3].strip()
    ]

    epoch_records.append({
        "Nomor_Baris": nomor_baris,
        "Waktu": waktu,
        "Epoch_Flag": epoch_flag,
        "Jumlah_Satelit_Tertulis": jumlah_sat,
        "Jumlah_Satelit_Terbaca": len(satelit),
        "Daftar_Satelit": satelit
    })

df_epoch_audit = pd.DataFrame(epoch_records)

print("RINGKASAN AUDIT EPOCH")
print("Jumlah epoch mentah:", len(df_epoch_audit))
print("Jumlah epoch GeoRinex:", obs.sizes["time"])

print("\nDISTRIBUSI EPOCH FLAG")
print(df_epoch_audit["Epoch_Flag"].value_counts().sort_index())

print("\nDISTRIBUSI JUMLAH SATELIT DALAM EPOCH RECORD")
print(
    df_epoch_audit["Jumlah_Satelit_Tertulis"]
    .value_counts()
    .sort_index()
)

print("\nPEMERIKSAAN JUMLAH SATELIT")
print(
    "Jumlah epoch dengan jumlah satelit tidak sesuai:",
    (
        df_epoch_audit["Jumlah_Satelit_Tertulis"]
        != df_epoch_audit["Jumlah_Satelit_Terbaca"]
    ).sum()
)

print("\nLIMA EPOCH PERTAMA")
display(df_epoch_audit.head())

print("\nLIMA EPOCH TERAKHIR")
display(df_epoch_audit.tail())

RINGKASAN AUDIT EPOCH
Jumlah epoch mentah: 2878
Jumlah epoch GeoRinex: 2878

DISTRIBUSI EPOCH FLAG
Epoch_Flag
0    2878
Name: count, dtype: int64

DISTRIBUSI JUMLAH SATELIT DALAM EPOCH RECORD
Jumlah_Satelit_Tertulis
7     177
8    1294
9    1407
Name: count, dtype: int64

PEMERIKSAAN JUMLAH SATELIT
Jumlah epoch dengan jumlah satelit tidak sesuai: 0

LIMA EPOCH PERTAMA


,Nomor_Baris,Waktu,Epoch_Flag,Jumlah_Satelit_Tertulis,Jumlah_Satelit_Terbaca,Daftar_Satelit
0,19,2003-10-27 00:00:30.000,0,9,9,"[G21, G06, G05, G25, G15, G18, G30, G14, G23]"
1,38,2003-10-27 00:01:00.000,0,9,9,"[G21, G06, G05, G25, G15, G18, G30, G14, G23]"
2,57,2003-10-27 00:01:30.000,0,9,9,"[G21, G06, G05, G25, G15, G18, G30, G14, G23]"
3,76,2003-10-27 00:02:00.001,0,9,9,"[G21, G06, G05, G25, G15, G18, G30, G14, G23]"
4,95,2003-10-27 00:02:30.001,0,9,9,"[G21, G06, G05, G25, G15, G18, G30, G14, G23]"



LIMA EPOCH TERAKHIR


,Nomor_Baris,Waktu,Epoch_Flag,Jumlah_Satelit_Tertulis,Jumlah_Satelit_Terbaca,Daftar_Satelit
2873,51310,2003-10-27 23:57:00.128,0,9,9,"[G25, G14, G21, G30, G15, G06, G23, G18, G05]"
2874,51329,2003-10-27 23:57:30.128,0,9,9,"[G25, G14, G21, G30, G15, G06, G23, G18, G05]"
2875,51348,2003-10-27 23:58:00.128,0,9,9,"[G25, G14, G21, G30, G15, G06, G23, G18, G05]"
2876,51367,2003-10-27 23:58:30.128,0,9,9,"[G25, G14, G21, G30, G15, G06, G23, G18, G05]"
2877,51386,2003-10-27 23:59:00.128,0,9,9,"[G25, G14, G21, G30, G15, G06, G23, G18, G05]"
